# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
#spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


**Data preparation**

In [3]:
#drop rows containing any missing value (be careful, create a new df otherwise this does not work)
spaceship_clean = spaceship.dropna(axis="rows")
#replace cabin values
spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]
#Drop PassengerId and Name
spaceship_clean = spaceship_clean.drop(columns = ["PassengerId","Name"])
# all columns
features = spaceship_clean.drop(columns=['Transported'], axis=1)
target = spaceship_clean["Transported"]

#here we make sure  that we gather all unique categorical values from our df columns in 'categories'. This way we avoid any missing and/or additional value
ohe = OneHotEncoder(sparse_output=False, drop='first', categories = [spaceship_clean[col].unique().tolist() for col in spaceship_clean.select_dtypes(include="object").columns])

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_28356\3849248933.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]


**Perform Train Test Split**

In [4]:
#Perform Train Test Split
X_train, X_test, y_train, y_test = train_test_split(features,target, test_size = 0.20, random_state=0)

# X train dummies for categorical columns
X_train_cat = X_train.select_dtypes(include="object") 
X_test_cat = X_test.select_dtypes(include="object") 
ohe.fit(X_train_cat)  
X_train_cat_trans_np = ohe.transform(X_train_cat)
X_train_cat_trans_df = pd.DataFrame(X_train_cat_trans_np, columns=ohe.get_feature_names_out(), index=X_train_cat.index)

X_test_cat_trans_np = ohe.transform(X_test_cat)
X_test_cat_trans_df = pd.DataFrame(X_test_cat_trans_np, columns=ohe.get_feature_names_out(), index=X_test_cat.index)

# X train for numerical columns
X_train_num = X_train.select_dtypes(include="number") 
X_test_num = X_test.select_dtypes(include="number") 

# concatenation of X_cat & X_num before scaling
X_train_transformed = pd.concat([X_train_num.reset_index(drop=True), pd.DataFrame(X_train_cat_trans_df).reset_index(drop=True)], axis = 1)
X_test_transformed = pd.concat([X_test_num.reset_index(drop=True), pd.DataFrame(X_test_cat_trans_df).reset_index(drop=True)], axis = 1)

# apply scaling for all
# creation of the instance
std_scaler = StandardScaler()

# application to the values
std_scaler.fit(X_train_transformed)

# transformation of the data
X_train_transformed_np = std_scaler.transform(X_train_transformed)
X_test_transformed_np  = std_scaler.transform(X_test_transformed)

# conversion into df
X_train_transformed_np_df = pd.DataFrame(X_train_transformed_np, columns=X_train_transformed.columns, index=X_train_transformed.index)
X_test_transformed_np_df  = pd.DataFrame(X_test_transformed_np, columns=X_test_transformed.columns, index=X_test_transformed.index)



**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [5]:
# Just for baseline, our current best model is a Decision Tree with R-Squared of 0.70, lets see how ensembles works
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100, # number of models to use
                               max_samples = 1000)

# Training Bagging model with our normalized data
bagging_reg.fit(X_train_transformed_np_df, y_train)

y_pred_test_bag = bagging_reg.predict(X_test_transformed_np_df)

print(f"MAE {mean_absolute_error(y_pred_test_bag, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"R2 score {bagging_reg.score(X_test_transformed_np_df, y_test): .2f}")

MAE  0.28
MSE  0.15
RMSE  0.38
R2 score  0.42


Despite the error metrics indicate a relative low rate of errors, the R2 Score (42%) is really under the base model R2 (70%), which suggest it does not add any value in this specific context.

- Random Forests

In [6]:
# Initialize a Random Forest
forest = RandomForestRegressor(n_estimators=100,
                             max_depth=20)

# Train the model
forest.fit(X_train_transformed_np_df, y_train)

# Evaluate
y_pred_test_bag = forest.predict(X_test_transformed_np_df)

print(f"MAE {mean_absolute_error(y_pred_test_bag, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"R2 score {forest.score(X_test_transformed_np_df, y_test): .2f}")

MAE  0.27
MSE  0.15
RMSE  0.38
R2 score  0.41


Despite the error metrics indicate a relative low rate of errors, the R2 Score (41%) is really under the base model R2 (70%), which suggest it does not add any value in this specific context.

- Gradient Boosting

In [7]:
# init
gb_reg = GradientBoostingRegressor(max_depth=20,
                                   n_estimators=100)

# train
gb_reg.fit(X_train_transformed_np_df, y_train)

# judge
y_pred_test_gb = gb_reg.predict(X_test_transformed_np_df)

print(f"MAE, {mean_absolute_error(y_pred_test_gb, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"R2 score, {gb_reg.score(X_test_transformed_np_df, y_test): .2f}")

MAE,  0.26
MSE,  0.17
RMSE,  0.41
R2 score,  0.33


**It is just getting worst!**

- Adaptive Boosting

In [8]:
ada_reg = AdaBoostRegressor(DecisionTreeRegressor(max_depth=20),
                            n_estimators=100)

ada_reg.fit(X_train_transformed_np_df, y_train)

y_pred_test_ada = ada_reg.predict(X_test_transformed_np_df)

print(f"MAE, {mean_absolute_error(y_pred_test_ada, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_ada, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_ada, y_test): .2f}")
print(f"R2 score, {ada_reg.score(X_test_transformed_np_df, y_test): .2f}")

MAE,  0.25
MSE,  0.18
RMSE,  0.43
R2 score,  0.28


Which model is the best and why?

In [ ]:
#For now the best model is the base model and i dont understand how this is possible! and regarding my previous work this would be the KNN model